In [4]:
# linkedin_assistant.ipynb
import os

os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"


# Import necessary libraries including LangChain's WebBaseLoader
from langchain.document_loaders import WebBaseLoader

# LinkedIn company profile URL
company_url = "https://www.linkedin.com/company/spectovx/"

# Initialize the loader with the URL
loader = WebBaseLoader(company_url)

# Load documents from the web page
docs = loader.load()

# Display preview of the extracted content
for i, doc in enumerate(docs):
    print(f"--- Document {i+1} ---")
    print(doc.page_content[:500])  # Print first 500 characters for preview
    print("\n")


--- Document 1 ---






 
















SpectoV | LinkedIn














 




      Skip to main content
    



LinkedIn
 








        Top Content
      







        People
      







        Learning
      







        Jobs
      







        Games
      






      Join now
    

          Sign in
      
 

 







 




 







                      SpectoV
                      
                    

                    Technology, Information and Internet
                

            




In [5]:
import pandas as pd
df=pd.read_csv(r"C:\Users\rajro\some random project\linkedin_companies_dataset.csv")
df.head()

,Company,Home,About,Posts,Jobs,People,Insights
0,Google,Google - Innovate. Transform. Lead.,Transforming communication and collaboration t...,Announcing new AI-powered features for better ...,Product Manager (London),155954 employees,Recognized for best-in-class workplace diversi...
1,Microsoft,Microsoft - Innovate. Transform. Lead.,Committed to sustainable growth and technology...,Announcing new AI-powered features for better ...,Product Manager (London),"Over 10,000 staff worldwide",Selected by UN for global AI ethics initiative.
2,Amazon,Amazon - Innovate. Transform. Lead.,Delivering seamless digital experiences with a...,Product launch: Secure Cloud Data Platform now...,Product Manager (London),Data not available,Achieved $10B revenue milestone last quarter.
3,Apple,Apple - Innovate. Transform. Lead.,Committed to sustainable growth and technology...,Awarded 'Best Employer' in tech sector for 2025.,Marketing Lead (Berlin),185117 employees,Selected by UN for global AI ethics initiative.
4,Meta,Meta - Innovate. Transform. Lead.,"A global leader in cloud and AI, empowering in...",Product launch: Secure Cloud Data Platform now...,DevOps Engineer (Toronto),145242 employees,Recognized for best-in-class workplace diversi...


In [7]:
print(df.info())
print("\nSample 'About' text:\n", df['About'].iloc[0])
print("\nSample 'Insights' text:\n", df['Insights'].iloc)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Company   120 non-null    object
 1   Home      120 non-null    object
 2   About     120 non-null    object
 3   Posts     120 non-null    object
 4   Jobs      92 non-null     object
 5   People    120 non-null    object
 6   Insights  120 non-null    object
dtypes: object(7)
memory usage: 6.7+ KB
None

Sample 'About' text:
 Transforming communication and collaboration through next-gen technologies in mobility and cloud.

Sample 'Insights' text:


In [9]:
import pandas as pd
import re
from typing import List

# Sample function to clean text
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace/newlines
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags if any
    text = text.strip()
    return text

# Function to chunk text into smaller pieces of max_length (words or characters)
def chunk_text(text: str, max_length: int = 300) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_length):
        chunk = ' '.join(words[i:i + max_length])
        chunks.append(chunk)
    return chunks

# Example usage with your DataFrame (assuming df contains your data)
def preprocess_and_chunk(df: pd.DataFrame, text_columns=['About', 'Insights'], chunk_size=300):
    data = []
    for idx, row in df.iterrows():
        company = row['Company']
        for col in text_columns:
            raw_text = row[col]
            if pd.isna(raw_text) or not raw_text.strip():
                continue
            cleaned_text = clean_text(raw_text)
            chunks = chunk_text(cleaned_text, max_length=chunk_size)
            for i, chunk in enumerate(chunks):
                data.append({
                    'Company': company,
                    'Source': col,
                    'Chunk_Index': i,
                    'Text_Chunk': chunk
                })
    return pd.DataFrame(data)

# Assuming your dataframe is named df
chunked_df = preprocess_and_chunk(df)

# Display some chunked data example
print(chunked_df.head())


     Company    Source  Chunk_Index  \
0     Google     About            0   
1     Google  Insights            0   
2  Microsoft     About            0   
3  Microsoft  Insights            0   
4     Amazon     About            0   

                                          Text_Chunk  
0  Transforming communication and collaboration t...  
1  Recognized for best-in-class workplace diversi...  
2  Committed to sustainable growth and technology...  
3    Selected by UN for global AI ethics initiative.  
4  Delivering seamless digital experiences with a...  


In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-mpnet-base-v2")


c:\Users\rajro\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajro\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [11]:
def combine_columns(row, columns=['Company', 'Home', 'About', 'Insights']):
    combined_text = ''
    for col in columns:
        text = row[col] if pd.notna(row[col]) else ''
        combined_text += f"{text} "  # simple concatenation, add separators if desired
    return combined_text.strip()

# Apply combination
df['Combined_Text'] = df.apply(combine_columns, axis=1)

# Now clean and chunk as before but on 'Combined_Text'
def preprocess_and_chunk_combined(df, text_column='Combined_Text', chunk_size=300):
    data = []
    for idx, row in df.iterrows():
        company = row['Company']
        combined_text = row[text_column]
        if not combined_text.strip():
            continue
        cleaned_text = clean_text(combined_text)
        chunks = chunk_text(cleaned_text, max_length=chunk_size)
        for i, chunk in enumerate(chunks):
            data.append({
                'Company': company,
                'Chunk_Index': i,
                'Text_Chunk': chunk
            })
    return pd.DataFrame(data)

chunked_df = preprocess_and_chunk_combined(df)


In [13]:
texts = chunked_df['Text_Chunk'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)
chunked_df['Embedding'] = list(embeddings)


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]


In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec

# Replace with your API key and your chosen region/cloud
api_key = "pcsk_6Fs29f_9P573X2VuwQsHtJKz7L7hJcgAzvjzTjY7QnSkpRNhRK9fKkZLsWNdf6NHtfH7DX"
cloud = "aws"          # or "gcp", depending on selected provider
region = "us-east-1" # e.g., "us-west-2", "us-east-1", etc.

pc = Pinecone(api_key=api_key)

index_name = "langchainvector"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,                  # matches your embedding dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )

# Connect to index
index = pc.Index(index_name)


In [20]:
# Prepare vectors for upsert
vectors = []
for i, row in chunked_df.iterrows():
    vector_id = f"{row['Company']}_{row['Chunk_Index']}"  # Unique ID per chunk
    embedding = row['Embedding']
    metadata = {
        "company": row['Company'],
        "chunk_index": row['Chunk_Index']
        # Add more metadata if needed
    }
    vectors.append((vector_id, embedding.tolist(), metadata))

# Upsert vectors to Pinecone
index.upsert(vectors)


{'upserted_count': 120}

In [22]:
def semantic_search(query: str, top_k: int = 5):
    query_embedding = model.encode([query])[0]
    result = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )
    retrieved_texts = []
    for match in result['matches']:
        meta = match['metadata']
        # Assuming 'company' and 'chunk_index' in metadata
        # Retrieve actual text chunk from chunked_df for returned company & chunk_index
        company = meta['company']
        chunk_idx = meta['chunk_index']
        text_chunk = chunked_df[(chunked_df['Company'] == company) & (chunked_df['Chunk_Index'] == chunk_idx)]['Text_Chunk'].values[0]
        retrieved_texts.append(text_chunk)
    return retrieved_texts


In [29]:
import requests
import json

# Assume model is your sentence-transformers embedding model
# Assume index is Pinecone index connected as before
# Assume chunked_df contains your company chunks with 'Company', 'Chunk_Index', 'Text_Chunk'

def semantic_search(query: str, top_k: int = 5):
    # Embed the user query
    query_embedding = model.encode([query])[0]

    # Query Pinecone index
    result = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )

    retrieved_texts = []
    for match in result['matches']:
        meta = match['metadata']
        company = meta['company']
        chunk_idx = meta['chunk_index']
        # Retrieve chunk text from your data
        text_chunk = chunked_df[
            (chunked_df['Company'] == company) & (chunked_df['Chunk_Index'] == chunk_idx)
        ]['Text_Chunk'].values[0]
        retrieved_texts.append(text_chunk)

    return retrieved_texts

def generate_answer_ollama(question: str, context_chunks: list):
    context = "\n\n".join(context_chunks)
    prompt = f"Use the following context to answer the question.\n{context}\n\nQuestion: {question}\nAnswer:"

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama2",
            "prompt": prompt,
            "options": {
                "temperature": 0.2,
                "max_tokens": 256
            }
        },
        stream=True  # Important for streamed response
    )

    answer = ""
    for line in response.iter_lines():
        if line:
            try:
                data = json.loads(line.decode("utf-8"))
                if "response" in data:
                    answer += data["response"]
            except Exception:
                pass

    return answer.strip()

def answer_query_with_ollama(user_query: str):
    relevant_chunks = semantic_search(user_query)
    answer = generate_answer_ollama(user_query, relevant_chunks)
    return answer


# Example usage:
user_query = "What common services does Company google and openai provides"
result = answer_query_with_ollama(user_query)
print(result)


Based on the context provided, both Google and OpenAI provide cutting-edge platforms and expert services in the areas of cloud and artificial intelligence (AI).


In [ ]:
# Prepare vectors for upsert
vectors = []
for i, row in chunked_df.iterrows():
    vector_id = f"{row['Company']}_{row['Chunk_Index']}"  # Unique ID per chunk
    embedding = row['Embedding']
    metadata = {
        "company": row['Company'],
        "chunk_index": row['Chunk_Index']
        # Add more metadata if needed
    }
    vectors.append((vector_id, embedding.tolist(), metadata))

# Upsert vectors to Pinecone
index.upsert(vectors)


{'upserted_count': 120}

In [30]:
import streamlit as st

st.title("LinkedIn Company Assistant")

# User input
user_question = st.text_input("Ask a question about a company:", "")

# Show button and spinner during processing
if st.button("Get Answer") and user_question.strip():
    with st.spinner("Searching and generating answer..."):
        try:
            answer = answer_query_with_ollama(user_question)
            st.markdown("### Answer:")
            st.write(answer)

            # Optional: show retrieved context chunks
            show_context = st.checkbox("Show retrieved context")
            if show_context:
                chunks = semantic_search(user_question)
                for i, chunk in enumerate(chunks):
                    st.markdown(f"**Chunk {i+1}:**")
                    st.write(chunk)

        except Exception as e:
            st.error(f"Error: {e}")


2025-08-17 14:53:11.262 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.490 
  command:

    streamlit run C:\Users\rajro\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-08-17 14:53:11.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.492 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.494 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.495 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.495 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-17 14:53:11.499 Thre